# LiteFNO Training (Google Colab)

This notebook sets up the `litefno-repro` repository, downloads a dataset, 
preprocesses it, and runs a full training job with structured logging for paper-ready results.

**Tip:** Use a GPU runtime in Colab for faster training.


## 1. Clone the repository

If you opened this notebook directly from GitHub in Colab, the repo is already present. 
Otherwise, this cell will clone it into `/content`.


In [ ]:
from pathlib import Path

repo_dir = Path('/content/litefno-repro')
if not repo_dir.exists():
    !git clone https://github.com/jgx0/litefno-repro.git {repo_dir}
%cd /content/litefno-repro


## 2. Install dependencies

Colab already includes PyTorch, but you can install a specific CUDA build if needed.


In [ ]:
# Optional: install a CUDA-specific PyTorch build
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q -e .
!pip install -q the-well-download ipywidgets


## 3. Configure the run

Use the widgets below to select dataset/experiment configs, experiment metadata, training epochs, device, 
and whether to download or preprocess data. The run outputs are stored under the chosen output root.


In [ ]:
import ipywidgets as widgets
from pathlib import Path
from datetime import datetime
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

dataset_options = sorted(Path('configs/datasets').glob('*.yaml'))
experiment_options = sorted(Path('configs/experiments').glob('*.yaml'))
dataset_options = [str(path) for path in dataset_options]
experiment_options = [str(path) for path in experiment_options]

default_dataset = 'configs/datasets/gray_scott_reaction_diffusion.yaml'
if default_dataset not in dataset_options and dataset_options:
    default_dataset = dataset_options[0]

default_experiment = 'configs/experiments/litefno_gray_scott_reaction_diffusion.yaml'
if default_experiment not in experiment_options and experiment_options:
    default_experiment = experiment_options[0]

timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
default_run_name = f'litefno_{timestamp}'

dataset_dropdown = widgets.Dropdown(
    options=dataset_options,
    value=default_dataset if dataset_options else None,
    description='Dataset',
    layout=widgets.Layout(width='80%'),
)
experiment_dropdown = widgets.Dropdown(
    options=experiment_options,
    value=default_experiment if experiment_options else None,
    description='Experiment',
    layout=widgets.Layout(width='80%'),
)
run_name_text = widgets.Text(
    value=default_run_name,
    description='Run name',
    layout=widgets.Layout(width='80%'),
)
output_root_text = widgets.Text(
    value='outputs/runs',
    description='Output root',
    layout=widgets.Layout(width='80%'),
)
epochs_slider = widgets.IntSlider(
    value=500,
    min=1,
    max=1000,
    step=1,
    description='Epochs',
    continuous_update=False,
)
device_dropdown = widgets.Dropdown(
    options=[('Auto', 'auto'), ('CPU', 'cpu'), ('CUDA', 'cuda')],
    value='auto',
    description='Device',
)
checkpoint_every_slider = widgets.IntSlider(
    value=50,
    min=0,
    max=500,
    step=10,
    description='Checkpoint every',
    continuous_update=False,
)
download_checkbox = widgets.Checkbox(value=True, description='Download data')
preprocess_checkbox = widgets.Checkbox(value=True, description='Preprocess data')
resume_text = widgets.Text(value='', description='Resume from', placeholder='/path/to/epoch_0100.pt')

display(widgets.VBox([
    dataset_dropdown,
    experiment_dropdown,
    run_name_text,
    output_root_text,
    epochs_slider,
    device_dropdown,
    checkpoint_every_slider,
    download_checkbox,
    preprocess_checkbox,
    resume_text,
]))


## 4. Resolve run settings

Run this cell after updating the widget values above to prepare output paths and overrides.


In [ ]:
import json
import shlex
import subprocess
from datetime import datetime
from pathlib import Path

def run(cmd_parts):
    cmd = ' '.join(shlex.quote(part) for part in cmd_parts)
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, check=True)

def clean_run_name(name: str) -> str:
    cleaned = name.strip().replace(' ', '_')
    if cleaned:
        return cleaned
    return f"litefno_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"

if not dataset_dropdown.value:
    raise ValueError('No dataset config found in configs/datasets.')
if not experiment_dropdown.value:
    raise ValueError('No experiment config found in configs/experiments.')

run_name = clean_run_name(run_name_text.value)
output_root = Path(output_root_text.value)
run_dir = output_root / run_name
run_dir.mkdir(parents=True, exist_ok=True)

metrics_path = run_dir / 'metrics.jsonl'
checkpoint_dir = run_dir / 'checkpoints'
epochs = int(epochs_slider.value)
checkpoint_every = int(checkpoint_every_slider.value)

overrides = [
    f'training.epochs={epochs}',
    f'logging.metrics_path={metrics_path}',
]

if checkpoint_every > 0:
    overrides += [
        f'training.checkpoint_dir={checkpoint_dir}',
        f'training.checkpoint_every={checkpoint_every}',
    ]

manifest = {
    'created_at': datetime.utcnow().isoformat() + 'Z',
    'dataset_config': dataset_dropdown.value,
    'experiment_config': experiment_dropdown.value,
    'overrides': overrides,
}
(run_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))

print('Run directory:', run_dir)
print('Metrics path:', metrics_path)
if checkpoint_every > 0:
    print('Checkpoint dir:', checkpoint_dir, f'(every {checkpoint_every} epochs)')
else:
    print('Checkpointing: disabled')


## 5. Download and preprocess data

Run this cell after updating the widget values above.


In [ ]:
if dataset_dropdown.value:
    if download_checkbox.value:
        run(['litefno', 'download', '--config', dataset_dropdown.value])
    if preprocess_checkbox.value:
        run(['litefno', 'preprocess', '--config', dataset_dropdown.value])
else:
    raise ValueError('No dataset config found in configs/datasets.')


## 6. Train LiteFNO

Run the full training loop. Metrics are logged to the per-run JSONL file prepared above.


In [ ]:
import torch

device_choice = device_dropdown.value
if device_choice == 'auto':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
else:
    device = device_choice

train_overrides = list(overrides)
train_overrides.append(f'training.device={device}')

resume_from = resume_text.value.strip()
if resume_from:
    train_overrides.append(f'training.resume_from={resume_from}')

train_cmd = [
    'litefno',
    'train',
    '--config',
    experiment_dropdown.value,
]
for override in train_overrides:
    train_cmd += ['--set', override]

run(train_cmd)


## 7. Review results

Load the JSONL log and summarize the best validation metrics for reporting.


In [ ]:
import json
from pathlib import Path

metrics_file = Path(metrics_path)
if not metrics_file.exists():
    raise FileNotFoundError(f'Metrics file not found: {metrics_file}')

records = [json.loads(line) for line in metrics_file.read_text().splitlines() if line.strip()]
if not records:
    raise ValueError('No metrics were logged. Check that training completed successfully.')

print(f'Loaded {len(records)} records from {metrics_file}')

def best_metric(key: str):
    values = [record[key] for record in records if key in record and isinstance(record[key], (int, float))]
    if not values:
        return None
    return min(values)

metric_keys = sorted({
    key for record in records for key in record
    if key not in {'step', 'params'}
})

best_metrics = {key: best_metric(key) for key in metric_keys}
final_record = records[-1]

summary = {
    'records': len(records),
    'final_step': final_record.get('step'),
    'final_metrics': final_record,
    'best_metrics': best_metrics,
}
summary_path = Path(run_dir) / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2))

print('Summary written to:', summary_path)
summary
